In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/LearnSyncAI")
DATA_DIR = PROJECT_DIR / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Files found:", list(DATA_DIR.iterdir()))


Project directory: /content/drive/MyDrive/LearnSyncAI
Data directory: /content/drive/MyDrive/LearnSyncAI/data
Files found: [PosixPath('/content/drive/MyDrive/LearnSyncAI/data/distribution_per_question.csv'), PosixPath('/content/drive/MyDrive/LearnSyncAI/data/tst_questions.csv'), PosixPath('/content/drive/MyDrive/LearnSyncAI/data/java_submissions.csv')]


In [5]:

import pandas as pd

file_path = DATA_DIR / "tst_questions.csv"

try:
    df_test = pd.read_csv(
        file_path,
        engine="python",
        sep=None,
        encoding="utf-8-sig"
    )

    print("Loaded successfully!")
    print("Shape:", df_test.shape)
    print("Columns:", df_test.columns.tolist())
    display(df_test.head(5))

except Exception as e:
    print("Still unable to load:", e)


Loaded successfully!
Shape: (99, 3)
Columns: ['question', 'statement', 'tests']


,question,statement,tests
0,bst_height,# BST Height\n\n The height of a tree is t...,['input: |\n 10 8 1 9 20 18 25\n out...
1,tree_height,# Height of a Tree\n \n Although trees s...,['input: |\n 1\n output: |\n ...
2,similar_trees,# Similar trees\n\n Two trees are said to b...,['input: |\n 3\n 2\n 1\n 3...
3,balance,# Balance in Binary Search Trees\n\n *The b...,['input: |\n 100 200 210 90 95 70 1\n ...
4,bfs,# Width Forwarding (BFS)\n\n There are seve...,['input: |\n 9\n output: |\n ...


In [6]:

file_path = DATA_DIR / "java_submissions.csv"

df_submissions = pd.read_csv(file_path, low_memory=False)

print("Shape:", df_submissions.shape)
print("\nColumns:")
print(df_submissions.columns.tolist())

print("\nData types:")
print(df_submissions.dtypes)

print("\nFirst 5 records:")
display(df_submissions.head())

print("\nMissing values:")
display(df_submissions.isnull().sum())

print("\nDuplicate rows:", df_submissions.duplicated().sum())


Shape: (16074, 9)

Columns:
['index', 'user', 'version', 'activity', 'timestamp', 'code', 'result', 'timisOrderedmp', 'timismp']

Data types:
index               int64
user               object
version             int64
activity           object
timestamp         float64
code               object
result             object
timisOrderedmp    float64
timismp           float64
dtype: object

First 5 records:


,index,user,version,activity,timestamp,code,result,timisOrderedmp,timismp
0,0,user_0001,1,negative_counting_sort,1.496324e+12,import java.util.Arrays;\nimport java.util.Sca...,success: ....,NaN,NaN
1,1,user_0001,1,order_statistic,1.496324e+12,import java.util.Scanner;\nclass OrderStatisti...,success: ....,NaN,NaN
2,2,user_0002,1,order_statistic,1.496368e+12,import java.util.Scanner;\nclass OrderStatisti...,success: ....,NaN,NaN
3,3,user_0003,1,order_statistic,1.496369e+12,import java.util.Scanner;\nclass OrderStatisti...,success: ....,NaN,NaN
4,4,user_0003,1,negative_counting_sort,1.496369e+12,import java.util.Arrays;\nimport java.util.Sca...,success: ....,NaN,NaN



Missing values:


,0
index,0
user,0
version,0
activity,0
timestamp,381
code,0
result,0
timisOrderedmp,15764
timismp,16003



Duplicate rows: 0


In [ ]:
#actual shit

In [13]:

import pandas as pd
import numpy as np

# Always start from the original raw dataset
df = df_submissions.copy()

# 1. Combine timestamp columns
timestamp_columns = [
    "timestamp",
    "timisOrderedmp",
    "timismp"
]

df["timestamp_unified"] = df[timestamp_columns].bfill(axis=1).iloc[:, 0]

# Identify timestamp source
df["timestamp_source"] = np.select(
    [
        df["timestamp"].notna(),
        df["timisOrderedmp"].notna(),
        df["timismp"].notna()
    ],
    [
        "timestamp",
        "timisOrderedmp",
        "timismp"
    ],
    default="missing"
)

# 2. Convert to datetime
df["timestamp_unified"] = pd.to_datetime(
    df["timestamp_unified"],
    unit="ms",
    errors="coerce"
)

# 3. Create binary outcomes
result_text = df["result"].astype(str).str.strip()

df["outcome"] = np.select(
    [
        result_text.str.startswith("success:"),
        result_text.str.startswith("fail:")
    ],
    [1, 0],
    default=-1
)

# 4. Retain valid binary observations with timestamps
processed = df[
    df["outcome"].isin([0, 1]) &
    df["timestamp_unified"].notna()
].copy()

# 5. Rename only the required columns
processed = processed.rename(columns={
    "user": "student_id",
    "activity": "question_id"
})

# 6. Create concept placeholder
processed["concept_id"] = "unmapped"

# 7. Select columns explicitly.
# This avoids retaining the original timestamp column twice.
processed = processed[
    [
        "student_id",
        "question_id",
        "concept_id",
        "timestamp_unified",
        "outcome",
        "result",
        "version",
        "timestamp_source"
    ]
].rename(columns={
    "timestamp_unified": "timestamp"
})

# 8. Sort chronologically
processed = processed.sort_values(
    ["student_id", "timestamp"]
).reset_index(drop=True)

# 9. Inspect
print("Processed dataset shape:", processed.shape)

print("\nOutcome distribution:")
print(processed["outcome"].value_counts())

print("\nTimestamp range:")
print(processed["timestamp"].min())
print(processed["timestamp"].max())

print("\nMissing values:")
print(processed.isnull().sum())

display(processed.head(10))

# 10. Save draft
output_path = DATA_DIR / "preprocessed_interactions_draft.csv"
processed.to_csv(output_path, index=False)

print("\nSaved:", output_path)


Processed dataset shape: (15474, 8)

Outcome distribution:
outcome
0    8935
1    6539
Name: count, dtype: int64

Timestamp range:
2017-05-16 14:20:33.791000
2024-09-12 02:51:12.630000

Missing values:
student_id          0
question_id         0
concept_id          0
timestamp           0
outcome             0
result              0
version             0
timestamp_source    0
dtype: int64


,student_id,question_id,concept_id,timestamp,outcome,result,version,timestamp_source
0,user_0001,duplicate_elements,unmapped,2017-05-16 14:20:33.791,0,fail: sssss,1,timestamp
1,user_0001,duplicate_elements,unmapped,2017-05-16 14:26:39.723,0,fail: sssss,2,timestamp
2,user_0001,recursive_power,unmapped,2017-05-16 14:36:29.021,0,fail: sssss,1,timestamp
3,user_0001,duplicate_elements,unmapped,2017-05-16 15:35:03.564,0,fail: eeeee,3,timestamp
4,user_0001,duplicate_elements,unmapped,2017-05-16 15:35:04.917,0,fail: eeeee,4,timestamp
5,user_0001,master_theorem,unmapped,2017-05-20 17:41:08.406,1,success: .....,1,timestamp
6,user_0001,master_theorem,unmapped,2017-05-20 17:41:09.946,1,success: .....,2,timestamp
7,user_0001,sorted_insertion_first,unmapped,2017-05-27 15:03:15.253,0,fail: FFF,1,timestamp
8,user_0001,sorted_insertion_first,unmapped,2017-05-27 15:04:56.206,0,fail: FFF,2,timestamp
9,user_0001,sorted_insertion_first,unmapped,2017-05-27 15:42:26.932,0,fail: ..e,3,timestamp



Saved: /content/drive/MyDrive/LearnSyncAI/data/preprocessed_interactions_draft.csv


In [16]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

# Load the question dataset using its actual delimiter
df_questions = pd.read_csv(
    DATA_DIR / "tst_questions.csv",
    sep="~",
    engine="python",
    encoding="utf-8-sig"
)

print("Question dataset loaded!")
print("Shape:", df_questions.shape)
print("Columns:", df_questions.columns.tolist())

display(df_questions.head(3))

assert len(df_questions) == 99, "Expected 99 activities; check the file."
assert {"question", "statement", "tests"}.issubset(df_questions.columns)


Question dataset loaded!
Shape: (99, 3)
Columns: ['question', 'statement', 'tests']


,question,statement,tests
0,bst_height,# BST Height\n\n The height of a tree is t...,['input: |\n 10 8 1 9 20 18 25\n out...
1,tree_height,# Height of a Tree\n \n Although trees s...,['input: |\n 1\n output: |\n ...
2,similar_trees,# Similar trees\n\n Two trees are said to b...,['input: |\n 3\n 2\n 1\n 3...


In [17]:
question_mapping = df_questions[
    ["question", "statement"]
].copy()

question_mapping["concept_id"] = "unmapped"
question_mapping["mapping_reason"] = ""

mapping_path = DATA_DIR / "concept_mapping_draft.csv"

question_mapping.to_csv(mapping_path, index=False)

print("Activities exported:", len(question_mapping))
print("Saved:", mapping_path)

display(question_mapping.head(10))


Activities exported: 99
Saved: /content/drive/MyDrive/LearnSyncAI/data/concept_mapping_draft.csv


,question,statement,concept_id,mapping_reason
0,bst_height,# BST Height\n\n The height of a tree is t...,unmapped,
1,tree_height,# Height of a Tree\n \n Although trees s...,unmapped,
2,similar_trees,# Similar trees\n\n Two trees are said to b...,unmapped,
3,balance,# Balance in Binary Search Trees\n\n *The b...,unmapped,
4,bfs,# Width Forwarding (BFS)\n\n There are seve...,unmapped,
5,right_bfs,# Right-to-left breadth-first search (BFS)\n\...,unmapped,
6,bubble_sort_step_by_step,# Bubble step by step\n \n Implement Bu...,unmapped,
7,build_heap,# Build Heap\n\n Write a program that reads...,unmapped,
8,binary_search,# Binary Search\n\n Implement a program tha...,unmapped,
9,recursive_binary_search,# Recursive Binary Search\n\n Implement a p...,unmapped,


In [20]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

df_questions = pd.read_csv(
    DATA_DIR / "tst_questions.csv",
    sep="~",
    engine="python",
    encoding="utf-8-sig"
)

# Initial mapping rules based on activity names
def suggest_mapping(question, statement):
    q = str(question).lower()
    s = str(statement).lower()
    text = q + " " + s

    # Explicit data structures and algorithms
    if any(x in q for x in ["bst", "_bst", "binary_search_tree"]):
        return "binary_search_trees", "high", "Activity explicitly concerns a Binary Search Tree."

    if "bfs" in q or "breadth-first" in s:
        return "bfs", "high", "Activity explicitly implements Breadth-First Search."

    if "dfs" in q or "depth-first" in s:
        return "dfs", "high", "Activity explicitly implements Depth-First Search."

    if "heap" in q or "heap" in s:
        return "heaps", "high", "Activity explicitly concerns heaps."

    if any(x in q for x in ["stack", "brackets", "reverse_polish"]):
        return "stacks", "high", "Activity explicitly uses a stack."

    if "queue" in q or "queue" in s:
        return "queues", "high", "Activity explicitly uses a queue."

    if "linked_list" in q or "linkedlist" in q:
        return "linked_lists", "high", "Activity explicitly concerns linked lists."

    if any(x in q for x in ["hash", "frequency_table", "frequency_element"]):
        return "hashing", "medium", "Activity concerns hashing or frequency-based lookup."

    if "graph" in q or "adjacency" in q or "tree" in q:
        return "trees" if "tree" in q else "graphs", "medium", "Activity concerns trees or graph representation."

    if "sort" in q or "partition" in q or "pivot" in q or "radix" in q:
        return "sorting", "high", "Activity concerns sorting or partitioning."

    if any(x in q for x in ["recursive", "recursion", "fibonacci", "tower_of_hanoi", "master_theorem"]):
        return "recursion", "medium", "Activity explicitly requires or focuses on recursion."

    if "binary_search" in q or "search" in q or "find_element" in q:
        return "arrays", "medium", "Sequence-search activity; arrays is the closest approved concept."

    if "array" in q or "vector" in q:
        return "arrays", "medium", "Activity explicitly concerns arrays or vectors."

    return "unmapped", "low", "No sufficiently clear match to an approved concept."

suggestions = df_questions.apply(
    lambda row: suggest_mapping(row["question"], row["statement"]),
    axis=1
)

df_questions["suggested_concept_id"] = [x[0] for x in suggestions]
df_questions["confidence"] = [x[1] for x in suggestions]
df_questions["mapping_reason"] = [x[2] for x in suggestions]

output_path = DATA_DIR / "concept_mapping_review.csv"
df_questions.to_csv(output_path, index=False)

print("Mapping worksheet created!")
print("Total activities:", len(df_questions))
print("\nSuggested concept distribution:")
print(df_questions["suggested_concept_id"].value_counts())

print("\nLow-confidence mappings:")
display(
    df_questions[
        df_questions["confidence"] == "low"
    ][["question", "suggested_concept_id", "mapping_reason"]]
)

print("\nSaved:", output_path)


Mapping worksheet created!
Total activities: 99

Suggested concept distribution:
suggested_concept_id
unmapped               23
sorting                20
binary_search_trees    14
stacks                 10
recursion               7
hashing                 4
linked_lists            4
arrays                  4
queues                  4
trees                   3
bfs                     2
heaps                   2
graphs                  2
Name: count, dtype: int64

Low-confidence mappings:


,question,suggested_concept_id,mapping_reason
3,balance,unmapped,No sufficiently clear match to an approved con...
13,count_leaves,unmapped,No sufficiently clear match to an approved con...
17,two_add_up,unmapped,No sufficiently clear match to an approved con...
18,down_n,unmapped,No sufficiently clear match to an approved con...
20,leftover_element,unmapped,No sufficiently clear match to an approved con...
21,duplicate_elements,unmapped,No sufficiently clear match to an approved con...
24,find_first_negative,unmapped,No sufficiently clear match to an approved con...
27,order_statistic,unmapped,No sufficiently clear match to an approved con...
28,mathematical_expression,unmapped,No sufficiently clear match to an approved con...
39,invert_phrase,unmapped,No sufficiently clear match to an approved con...



Saved: /content/drive/MyDrive/LearnSyncAI/data/concept_mapping_review.csv


In [21]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

mapping_path = DATA_DIR / "concept_mapping_review.csv"
df_mapping = pd.read_csv(mapping_path)

# Manually reviewed corrections
corrections = {
    "balance": ("binary_search_trees", "high",
                "Calculates balance factors in a BST."),

    "count_leaves": ("binary_search_trees", "high",
                     "Counts leaf nodes in a BST."),

    "two_add_up": ("arrays", "high",
                   "Finds two values in an integer array."),

    "down_n": ("arrays", "high",
               "Selects the n smallest elements from a sequence."),

    "leftover_element": ("arrays", "high",
                         "Compares two integer arrays."),

    "duplicate_elements": ("arrays", "high",
                           "Checks duplicates using an array."),

    "find_first_negative": ("recursion", "high",
                            "The statement explicitly requires recursion."),

    "move_impostor": ("sorting", "high",
                      "Repositions an out-of-order element."),

    "move_n": ("sorting", "high",
               "Repositions multiple out-of-order elements."),

    "move_three": ("sorting", "high",
                   "Repositions three out-of-order elements."),

    "element_positions": ("arrays", "high",
                          "Finds positions of an element in a sequence."),

    "sum_leaves": ("binary_search_trees", "high",
                   "Calculates the sum of BST leaf values."),

    "swap_neighbors": ("arrays", "high",
                       "Swaps neighbouring array elements."),

    "warm_up_dsa": ("arrays", "high",
                    "Performs element-wise multiplication on a sequence."),

    "check_tree": ("graphs", "high",
                   "Checks connectivity and cycles in an undirected graph.")
}

for question, (concept, confidence, reason) in corrections.items():
    mask = df_mapping["question"] == question

    if mask.any():
        df_mapping.loc[mask, "suggested_concept_id"] = concept
        df_mapping.loc[mask, "confidence"] = confidence
        df_mapping.loc[mask, "mapping_reason"] = reason

# Keep uncertain activities unmapped
uncertain = [
    "mathematical_expression",
    "invert_phrase",
    "mariana_and_books",
    "stopwords",
    "top_n",
    "three_n_plus_one",
    "word_cloud",
    "order_statistic"
]

for question in uncertain:
    mask = df_mapping["question"] == question
    df_mapping.loc[mask, "suggested_concept_id"] = "unmapped"
    df_mapping.loc[mask, "confidence"] = "review"
    df_mapping.loc[mask, "mapping_reason"] = (
        "Requires further review; insufficient evidence for a "
        "confident approved-concept mapping."
    )

# Save revised worksheet
reviewed_path = DATA_DIR / "concept_mapping_review_v2.csv"
df_mapping.to_csv(reviewed_path, index=False)

print("Revised mapping saved:", reviewed_path)
print("\nConcept distribution:")
print(df_mapping["suggested_concept_id"].value_counts())

print("\nActivities requiring review:")
display(
    df_mapping[
        df_mapping["confidence"] == "review"
    ][["question", "suggested_concept_id", "mapping_reason"]]
)


Revised mapping saved: /content/drive/MyDrive/LearnSyncAI/data/concept_mapping_review_v2.csv

Concept distribution:
suggested_concept_id
sorting                23
binary_search_trees    17
arrays                 11
stacks                 10
unmapped                9
recursion               8
linked_lists            4
hashing                 4
queues                  4
graphs                  3
heaps                   2
trees                   2
bfs                     2
Name: count, dtype: int64

Activities requiring review:


,question,suggested_concept_id,mapping_reason
27,order_statistic,unmapped,Requires further review; insufficient evidence...
28,mathematical_expression,unmapped,Requires further review; insufficient evidence...
39,invert_phrase,unmapped,Requires further review; insufficient evidence...
46,mariana_and_books,unmapped,Requires further review; insufficient evidence...
83,stopwords,unmapped,Requires further review; insufficient evidence...
89,top_n,unmapped,Requires further review; insufficient evidence...
91,three_n_plus_one,unmapped,Requires further review; insufficient evidence...
98,word_cloud,unmapped,Requires further review; insufficient evidence...


In [22]:
# Identify every unmapped activity

unmapped = df_mapping[
    df_mapping["suggested_concept_id"] == "unmapped"
]

print("Total unmapped:", len(unmapped))

for _, row in unmapped.iterrows():
    print("\nActivity:", row["question"])
    print("Confidence:", row["confidence"])
    print("Reason:", row["mapping_reason"])
    print("Statement:", str(row["statement"])[:600])
    print("-" * 60)


Total unmapped: 9

Activity: order_statistic
Confidence: review
Reason: Requires further review; insufficient evidence for a confident approved-concept mapping.
Statement: # Order statistic

    The i-th order statistic of a sequence of integers is the i-th smallest element of that sequence. 
    For example, in the sequence [4,5,9,1,2,6,8], the number 4 is the third order statistic, 
    because 4 is the third smallest number. 

    Write a program that prints out the order statistic of the first element of a sequence.
    Although the tests only check that your program prints the order statistic of the first element, 
    at the end of the execution of your algorithm, the first element must divide the sequence in such a way that 
    that the elements smaller 
------------------------------------------------------------

Activity: mathematical_expression
Confidence: review
Reason: Requires further review; insufficient evidence for a confident approved-concept mapping.
Statement: # Ma

In [23]:
# Display all activity-to-concept assignments

review_table = df_mapping[
    ["question", "suggested_concept_id", "confidence"]
].sort_values(
    ["suggested_concept_id", "question"]
)

display(review_table)

print("\nActivities per concept:")
print(
    review_table["suggested_concept_id"]
    .value_counts()
    .sort_index()
)


,question,suggested_concept_id,confidence
8,binary_search,arrays,medium
96,circular_vector,arrays,medium
18,down_n,arrays,high
21,duplicate_elements,arrays,high
66,element_positions,arrays,high
...,...,...,...
83,stopwords,unmapped,review
91,three_n_plus_one,unmapped,review
89,top_n,unmapped,review
80,without_pair,unmapped,low



Activities per concept:
suggested_concept_id
arrays                 11
bfs                     2
binary_search_trees    17
graphs                  3
hashing                 4
heaps                   2
linked_lists            4
queues                  4
recursion               8
sorting                23
stacks                 10
trees                   2
unmapped                9
Name: count, dtype: int64


In [24]:

# Apply final decisions to the mapping worksheet

final_corrections = {
    "order_statistic": (
        "sorting",
        "high",
        "The problem requires partitioning a sequence around its first element."
    ),
    "without_pair": (
        "arrays",
        "high",
        "Explicitly identifies a unique-frequency element in an unordered array."
    ),
    "mariana_and_books": (
        "unmapped",
        "review",
        "Full problem statement must be inspected before assigning a concept."
    )
}

for question, (concept, confidence, reason) in final_corrections.items():
    mask = df_mapping["question"] == question

    df_mapping.loc[mask, "suggested_concept_id"] = concept
    df_mapping.loc[mask, "confidence"] = confidence
    df_mapping.loc[mask, "mapping_reason"] = reason

# Save the revised file
final_path = DATA_DIR / "concept_mapping_review_v3.csv"
df_mapping.to_csv(final_path, index=False)

print("Saved:", final_path)
print("\nFinal concept distribution:")
print(df_mapping["suggested_concept_id"].value_counts())

print("\nRemaining unmapped activities:")
display(
    df_mapping[
        df_mapping["suggested_concept_id"] == "unmapped"
    ][["question", "confidence", "mapping_reason"]]
)


Saved: /content/drive/MyDrive/LearnSyncAI/data/concept_mapping_review_v3.csv

Final concept distribution:
suggested_concept_id
sorting                24
binary_search_trees    17
arrays                 12
stacks                 10
recursion               8
unmapped                7
linked_lists            4
hashing                 4
queues                  4
graphs                  3
heaps                   2
trees                   2
bfs                     2
Name: count, dtype: int64

Remaining unmapped activities:


,question,confidence,mapping_reason
28,mathematical_expression,review,Requires further review; insufficient evidence...
39,invert_phrase,review,Requires further review; insufficient evidence...
46,mariana_and_books,review,Full problem statement must be inspected befor...
83,stopwords,review,Requires further review; insufficient evidence...
89,top_n,review,Requires further review; insufficient evidence...
91,three_n_plus_one,review,Requires further review; insufficient evidence...
98,word_cloud,review,Requires further review; insufficient evidence...


In [6]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

# 1. Load reviewed mapping
df_mapping = pd.read_csv(
    DATA_DIR / "concept_mapping_review_v3.csv"
)

df_mapping = df_mapping.rename(columns={
    "question": "question_id",
    "suggested_concept_id": "concept_id"
})

# 2. Validate mapping
assert len(df_mapping) == 99
assert df_mapping["question_id"].is_unique

# Save final mapping file
mapping_columns = [
    "question_id",
    "concept_id",
    "mapping_reason",
    "confidence"
]

df_mapping[mapping_columns].to_csv(
    DATA_DIR / "concept_mapping.csv",
    index=False
)

# 3. Load interaction draft
df_interactions = pd.read_csv(
    DATA_DIR / "preprocessed_interactions_draft.csv",
    parse_dates=["timestamp"]
)

# IMPORTANT: Remove the old placeholder concept_id
df_interactions = df_interactions.drop(
    columns=["concept_id"],
    errors="ignore"
)

# 4. Merge actual concept mappings
df_final = df_interactions.merge(
    df_mapping[mapping_columns],
    on="question_id",
    how="left",
    validate="many_to_one"
)

# 5. Validate mapping coverage
missing_mapping = df_final["concept_id"].isna()

print("Missing mapping records:", missing_mapping.sum())

assert missing_mapping.sum() == 0, (
    "Some question IDs were not found in the mapping file!"
)

# 6. Save final interactions
df_final.to_csv(
    DATA_DIR / "preprocessed_interactions.csv",
    index=False
)

# 7. Final report
print("\nFINAL DATASET")
print("Rows:", len(df_final))
print("Students:", df_final["student_id"].nunique())
print("Activities:", df_final["question_id"].nunique())

print("\nConcept distribution:")
print(df_final["concept_id"].value_counts())

print("\nMissing values:")
print(df_final.isnull().sum())

print("\nSaved successfully:")
print(DATA_DIR / "concept_mapping.csv")
print(DATA_DIR / "preprocessed_interactions.csv")

display(df_final.head(10))


Missing mapping records: 0

FINAL DATASET
Rows: 15474
Students: 270
Activities: 99

Concept distribution:
concept_id
sorting                4361
arrays                 2254
queues                 1675
binary_search_trees    1666
stacks                 1209
recursion              1114
unmapped               1084
hashing                 814
heaps                   455
linked_lists            385
bfs                     312
graphs                  124
trees                    21
Name: count, dtype: int64

Missing values:
student_id          0
question_id         0
timestamp           0
outcome             0
result              0
version             0
timestamp_source    0
concept_id          0
mapping_reason      0
confidence          0
dtype: int64

Saved successfully:
/content/drive/MyDrive/LearnSyncAI/data/concept_mapping.csv
/content/drive/MyDrive/LearnSyncAI/data/preprocessed_interactions.csv


,student_id,question_id,timestamp,outcome,result,version,timestamp_source,concept_id,mapping_reason,confidence
0,user_0001,duplicate_elements,2017-05-16 14:20:33.791,0,fail: sssss,1,timestamp,arrays,Checks duplicates using an array.,high
1,user_0001,duplicate_elements,2017-05-16 14:26:39.723,0,fail: sssss,2,timestamp,arrays,Checks duplicates using an array.,high
2,user_0001,recursive_power,2017-05-16 14:36:29.021,0,fail: sssss,1,timestamp,recursion,Activity explicitly requires or focuses on rec...,medium
3,user_0001,duplicate_elements,2017-05-16 15:35:03.564,0,fail: eeeee,3,timestamp,arrays,Checks duplicates using an array.,high
4,user_0001,duplicate_elements,2017-05-16 15:35:04.917,0,fail: eeeee,4,timestamp,arrays,Checks duplicates using an array.,high
5,user_0001,master_theorem,2017-05-20 17:41:08.406,1,success: .....,1,timestamp,recursion,Activity explicitly requires or focuses on rec...,medium
6,user_0001,master_theorem,2017-05-20 17:41:09.946,1,success: .....,2,timestamp,recursion,Activity explicitly requires or focuses on rec...,medium
7,user_0001,sorted_insertion_first,2017-05-27 15:03:15.253,0,fail: FFF,1,timestamp,sorting,Activity concerns sorting or partitioning.,high
8,user_0001,sorted_insertion_first,2017-05-27 15:04:56.206,0,fail: FFF,2,timestamp,sorting,Activity concerns sorting or partitioning.,high
9,user_0001,sorted_insertion_first,2017-05-27 15:42:26.932,0,fail: ..e,3,timestamp,sorting,Activity concerns sorting or partitioning.,high


In [7]:
#bkt

In [8]:
import pandas as pd
from collections import defaultdict
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

def build_student_sequences(interactions):
    """
    Build chronological interaction sequences for each
    student and DSA concept.

    Unmapped activities are excluded.
    """

    required_columns = {
        "student_id",
        "concept_id",
        "question_id",
        "timestamp",
        "outcome"
    }

    missing = required_columns - set(interactions.columns)

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    data = interactions.copy()

    data["timestamp"] = pd.to_datetime(
        data["timestamp"], errors="coerce"
    )

    data["outcome"] = pd.to_numeric(
        data["outcome"], errors="coerce"
    )

    # Retain only valid mapped observations
    data = data[
        data["concept_id"].notna() &
        (data["concept_id"] != "unmapped") &
        data["outcome"].isin([0, 1]) &
        data["timestamp"].notna()
    ].copy()

    # Sort each student's observations chronologically
    data = data.sort_values(
        ["student_id", "timestamp"]
    )

    sequences = defaultdict(list)

    for row in data.itertuples(index=False):
        key = (row.student_id, row.concept_id)

        sequences[key].append({
            "question_id": row.question_id,
            "timestamp": row.timestamp.isoformat(),
            "outcome": int(row.outcome)
        })

    return dict(sequences)


# Load final interactions
interactions = pd.read_csv(
    DATA_DIR / "preprocessed_interactions.csv",
    parse_dates=["timestamp"]
)

# Build sequences
student_sequences = build_student_sequences(interactions)

# Validate and inspect
print("Total student-concept sequences:", len(student_sequences))

print("Unique students with mapped interactions:",
      len(set(student_id for student_id, concept
              in student_sequences.keys())))

print("Total mapped observations:",
      sum(len(seq) for seq in student_sequences.values()))

print("\nExample sequences:")

for key, sequence in list(student_sequences.items())[:5]:
    print("\nStudent:", key[0])
    print("Concept:", key[1])
    print("Number of attempts:", len(sequence))
    print("First attempts:", sequence[:3])


Total student-concept sequences: 1508
Unique students with mapped interactions: 270
Total mapped observations: 14390

Example sequences:

Student: user_0001
Concept: arrays
Number of attempts: 4
First attempts: [{'question_id': 'duplicate_elements', 'timestamp': '2017-05-16T14:20:33.791000', 'outcome': 0}, {'question_id': 'duplicate_elements', 'timestamp': '2017-05-16T14:26:39.723000', 'outcome': 0}, {'question_id': 'duplicate_elements', 'timestamp': '2017-05-16T15:35:03.564000', 'outcome': 0}]

Student: user_0001
Concept: recursion
Number of attempts: 3
First attempts: [{'question_id': 'recursive_power', 'timestamp': '2017-05-16T14:36:29.021000', 'outcome': 0}, {'question_id': 'master_theorem', 'timestamp': '2017-05-20T17:41:08.406000', 'outcome': 1}, {'question_id': 'master_theorem', 'timestamp': '2017-05-20T17:41:09.946000', 'outcome': 1}]

Student: user_0001
Concept: sorting
Number of attempts: 21
First attempts: [{'question_id': 'sorted_insertion_first', 'timestamp': '2017-05-27T1

In [10]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

mapping = pd.read_csv(DATA_DIR / "concept_mapping.csv")

corrections = {
    "hash_table_chaining": "hashing",
    "queue_jumper": "arrays"
}

for question_id, new_concept in corrections.items():
    mapping.loc[
        mapping["question_id"] == question_id,
        "concept_id"
    ] = new_concept

mapping.to_csv(DATA_DIR / "concept_mapping.csv", index=False)

print("Corrections saved successfully!")
print(
    mapping[mapping["question_id"].isin(corrections)].to_string(index=False)
)

Corrections saved successfully!
        question_id concept_id                    mapping_reason confidence
       queue_jumper     arrays Activity explicitly uses a queue.       high
hash_table_chaining    hashing Activity explicitly uses a queue.       high


In [11]:
mapping = pd.read_csv(DATA_DIR / "concept_mapping.csv")

print("Total mapped activities:", len(mapping))
print("\nConcept distribution:")
print(mapping["concept_id"].value_counts())

print("\nUnmapped activities:")
print(mapping[mapping["concept_id"] == "unmapped"]["question_id"].tolist())

Total mapped activities: 99

Concept distribution:
concept_id
sorting                24
binary_search_trees    17
arrays                 13
stacks                 10
recursion               8
unmapped                7
hashing                 5
linked_lists            4
graphs                  3
heaps                   2
bfs                     2
trees                   2
queues                  2
Name: count, dtype: int64

Unmapped activities:
['mathematical_expression', 'invert_phrase', 'mariana_and_books', 'stopwords', 'top_n', 'three_n_plus_one', 'word_cloud']


In [12]:
# Correct the explanations for the two manually reviewed mappings

mapping.loc[
    mapping["question_id"] == "queue_jumper",
    "mapping_reason"
] = "Primarily involves array rearrangement and element movement."

mapping.loc[
    mapping["question_id"] == "hash_table_chaining",
    "mapping_reason"
] = "Implements a hash table using chaining for collision resolution."

mapping.to_csv(DATA_DIR / "concept_mapping.csv", index=False)

print(mapping[
    mapping["question_id"].isin(
        ["queue_jumper", "hash_table_chaining"]
    )
][["question_id", "concept_id", "mapping_reason"]].to_string(index=False))

        question_id concept_id                                                   mapping_reason
       queue_jumper     arrays     Primarily involves array rearrangement and element movement.
hash_table_chaining    hashing Implements a hash table using chaining for collision resolution.


In [13]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")
bkt_code = r'''
import math

# Baseline BKT parameters.
# These are configurable assumptions, not dataset-trained estimates.
P_INIT = 0.20
P_LEARN = 0.10
P_GUESS = 0.20
P_SLIP = 0.10

# Stores current mastery probabilities by (student_id, concept_id).
_mastery = {}


def _validate_parameters(p_init, p_learn, p_guess, p_slip):
    for name, value in {
        "p_init": p_init,
        "p_learn": p_learn,
        "p_guess": p_guess,
        "p_slip": p_slip
    }.items():
        if not 0 <= value <= 1:
            raise ValueError(f"{name} must be between 0 and 1.")

    if p_guess + p_slip >= 2:
        raise ValueError("Invalid guess/slip parameter combination.")


def _update_probability(prior, outcome, p_learn, p_guess, p_slip):
    if outcome == 1:
        numerator = prior * (1 - p_slip)
        denominator = numerator + (1 - prior) * p_guess
    else:
        numerator = prior * p_slip
        denominator = numerator + (1 - prior) * (1 - p_guess)

    posterior = numerator / denominator if denominator else prior

    # Account for the possibility of learning after the attempt.
    updated_mastery = posterior + (1 - posterior) * p_learn

    return posterior, updated_mastery


def update_bkt(student_id, concept_id, outcome):
    """
    Update a student's mastery estimate after a binary response.
    outcome: 1 = correct, 0 = incorrect.
    """
    if outcome not in (0, 1):
        raise ValueError("Outcome must be binary: 0 or 1.")

    key = (str(student_id), str(concept_id))
    prior = _mastery.get(key, P_INIT)

    predicted_correct = (
        prior * (1 - P_SLIP) +
        (1 - prior) * P_GUESS
    )

    posterior, updated = _update_probability(
        prior, outcome, P_LEARN, P_GUESS, P_SLIP
    )

    _mastery[key] = updated

    return {
        "student_id": str(student_id),
        "concept_id": str(concept_id),
        "outcome": int(outcome),
        "prior_mastery": prior,
        "predicted_correct_probability": predicted_correct,
        "posterior_mastery": posterior,
        "mastery": updated
    }


def get_mastery_profile(student_id):
    """Return the current mastery estimates for one student."""
    student_id = str(student_id)

    return {
        concept_id: mastery
        for (sid, concept_id), mastery in _mastery.items()
        if sid == student_id
    }


def predict_correct_probability(student_id, concept_id):
    """Predict the probability of a correct answer before the next attempt."""
    key = (str(student_id), str(concept_id))
    mastery = _mastery.get(key, P_INIT)

    return mastery * (1 - P_SLIP) + (1 - mastery) * P_GUESS


def reset_bkt_state():
    """Clear all in-memory mastery estimates."""
    _mastery.clear()


def evaluate_bkt(sequences):
    """
    Evaluate one-step-ahead predictions on chronological sequences.
    Each prediction is made before observing that attempt's outcome.
    """
    reset_bkt_state()

    actual = []
    predicted = []

    for (student_id, concept_id), attempts in sequences.items():
        for attempt in attempts:
            probability = predict_correct_probability(
                student_id, concept_id
            )

            actual.append(int(attempt["outcome"]))
            predicted.append(probability)

            update_bkt(
                student_id,
                concept_id,
                int(attempt["outcome"])
            )

    if not actual:
        raise ValueError("No observations available for evaluation.")

    epsilon = 1e-15
    clipped = [
        min(max(p, epsilon), 1 - epsilon)
        for p in predicted
    ]

    accuracy = sum(
        (p >= 0.5) == bool(y)
        for p, y in zip(predicted, actual)
    ) / len(actual)

    brier_score = sum(
        (p - y) ** 2
        for p, y in zip(predicted, actual)
    ) / len(actual)

    log_loss = -sum(
        y * math.log(p) + (1 - y) * math.log(1 - p)
        for p, y in zip(clipped, actual)
    ) / len(actual)

    return {
        "observations": len(actual),
        "accuracy": accuracy,
        "brier_score": brier_score,
        "log_loss": log_loss
    }
'''

(DATA_DIR / "bkt_engine.py").write_text(bkt_code, encoding="utf-8")

print("Created:", DATA_DIR / "bkt_engine.py")

Created: /content/drive/MyDrive/LearnSyncAI/data/bkt_engine.py


In [14]:
import sys

if str(DATA_DIR) not in sys.path:
    sys.path.append(str(DATA_DIR))

from bkt_engine import (
    update_bkt,
    get_mastery_profile,
    predict_correct_probability,
    reset_bkt_state,
    evaluate_bkt
)

reset_bkt_state()

print("Initial predicted correctness:",
      predict_correct_probability("user_0001", "arrays"))

print("\nAfter an incorrect answer:")
print(update_bkt("user_0001", "arrays", 0))

print("\nAfter a correct answer:")
print(update_bkt("user_0001", "arrays", 1))

print("\nMastery profile:")
print(get_mastery_profile("user_0001"))

Initial predicted correctness: 0.3400000000000001

After an incorrect answer:
{'student_id': 'user_0001', 'concept_id': 'arrays', 'outcome': 0, 'prior_mastery': 0.2, 'predicted_correct_probability': 0.3400000000000001, 'posterior_mastery': 0.030303030303030304, 'mastery': 0.1272727272727273}

After a correct answer:
{'student_id': 'user_0001', 'concept_id': 'arrays', 'outcome': 1, 'prior_mastery': 0.1272727272727273, 'predicted_correct_probability': 0.2890909090909091, 'posterior_mastery': 0.39622641509433965, 'mastery': 0.45660377358490567}

Mastery profile:
{'arrays': 0.45660377358490567}


In [15]:
from bkt_engine import evaluate_bkt, get_mastery_profile

# Evaluate BKT on the prepared chronological sequences
results = evaluate_bkt(student_sequences)

print("BKT EVALUATION RESULTS")
print("-" * 40)
print("Total observations:", results["observations"])
print("Prediction accuracy:", round(results["accuracy"], 4))
print("Brier score:", round(results["brier_score"], 4))
print("Log loss:", round(results["log_loss"], 4))

print("\nExample mastery profile after evaluation:")
print(get_mastery_profile("user_0001"))

BKT EVALUATION RESULTS
----------------------------------------
Total observations: 14390
Prediction accuracy: 0.6814
Brier score: 0.2233
Log loss: 0.6507

Example mastery profile after evaluation:
{'arrays': 0.11432102524959908, 'recursion': 0.8117647058823529, 'sorting': 0.9706093376811333, 'stacks': 0.9558958652373661, 'queues': 0.11686773911146048, 'hashing': 0.11611253196930947, 'binary_search_trees': 0.9996684052364342, 'bfs': 0.5764705882352941, 'heaps': 0.7980128579777908}


In [16]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

assessment_code = r'''
"""
LearnSync AI - Assessment Agent (Member 1)

Provides DSA multiple-choice assessment questions and
evaluates answers using deterministic answer matching.
"""

QUESTION_BANK = [
    {
        "question_id": "assess_arrays_01",
        "concept_id": "arrays",
        "question": "What is the time complexity of accessing an element using its index in an array?",
        "options": {
            "A": "O(1)",
            "B": "O(n)",
            "C": "O(log n)",
            "D": "O(n^2)"
        },
        "correct_answer": "A",
        "explanation": "Array elements can be accessed directly using their index."
    },
    {
        "question_id": "assess_linked_lists_01",
        "concept_id": "linked_lists",
        "question": "In a singly linked list, what does each node normally contain?",
        "options": {
            "A": "Only an array index",
            "B": "Data and a reference to the next node",
            "C": "Two child nodes",
            "D": "A priority value only"
        },
        "correct_answer": "B",
        "explanation": "A singly linked-list node stores data and a link to the next node."
    },
    {
        "question_id": "assess_stacks_01",
        "concept_id": "stacks",
        "question": "Which principle does a stack follow?",
        "options": {
            "A": "FIFO",
            "B": "Random access",
            "C": "LIFO",
            "D": "Priority-based access"
        },
        "correct_answer": "C",
        "explanation": "A stack follows Last In, First Out."
    },
    {
        "question_id": "assess_queues_01",
        "concept_id": "queues",
        "question": "Which principle does a standard queue follow?",
        "options": {
            "A": "LIFO",
            "B": "FIFO",
            "C": "Random access",
            "D": "Sorted access"
        },
        "correct_answer": "B",
        "explanation": "A standard queue follows First In, First Out."
    },
    {
        "question_id": "assess_recursion_01",
        "concept_id": "recursion",
        "question": "What is essential to prevent infinite recursive calls?",
        "options": {
            "A": "A hash function",
            "B": "A base case",
            "C": "A sorting operation",
            "D": "A queue"
        },
        "correct_answer": "B",
        "explanation": "A base case provides a stopping condition for recursion."
    },
    {
        "question_id": "assess_trees_01",
        "concept_id": "trees",
        "question": "In a tree data structure, what is the topmost node called?",
        "options": {
            "A": "Leaf",
            "B": "Sibling",
            "C": "Root",
            "D": "Edge"
        },
        "correct_answer": "C",
        "explanation": "The root is the topmost node of a tree."
    },
    {
        "question_id": "assess_bst_01",
        "concept_id": "binary_search_trees",
        "question": "In a binary search tree, values in the left subtree of a node are generally:",
        "options": {
            "A": "Greater than the node",
            "B": "Smaller than the node",
            "C": "Always equal to the node",
            "D": "Stored randomly"
        },
        "correct_answer": "B",
        "explanation": "In the standard BST ordering, left-subtree values are smaller than the node."
    },
    {
        "question_id": "assess_heaps_01",
        "concept_id": "heaps",
        "question": "In a max-heap, which element is at the root?",
        "options": {
            "A": "The smallest element",
            "B": "The largest element",
            "C": "The oldest element",
            "D": "A random element"
        },
        "correct_answer": "B",
        "explanation": "The maximum element is stored at the root of a max-heap."
    },
    {
        "question_id": "assess_sorting_01",
        "concept_id": "sorting",
        "question": "Which sorting algorithm repeatedly selects the minimum element from the unsorted portion?",
        "options": {
            "A": "Selection sort",
            "B": "Merge sort",
            "C": "Quick sort",
            "D": "Heap sort"
        },
        "correct_answer": "A",
        "explanation": "Selection sort repeatedly selects the minimum remaining element."
    },
    {
        "question_id": "assess_hashing_01",
        "concept_id": "hashing",
        "question": "What is the main purpose of a hash function in a hash table?",
        "options": {
            "A": "To sort all stored values",
            "B": "To map a key to an index or table position",
            "C": "To reverse a linked list",
            "D": "To traverse a graph"
        },
        "correct_answer": "B",
        "explanation": "A hash function maps a key to a table position."
    },
    {
        "question_id": "assess_graphs_01",
        "concept_id": "graphs",
        "question": "What does an edge represent in a graph?",
        "options": {
            "A": "A connection between vertices",
            "B": "The root of a tree",
            "C": "An array index",
            "D": "A sorting operation"
        },
        "correct_answer": "A",
        "explanation": "An edge represents a connection between vertices."
    },
    {
        "question_id": "assess_bfs_01",
        "concept_id": "bfs",
        "question": "Which data structure is commonly used to manage the frontier in Breadth-First Search?",
        "options": {
            "A": "Stack",
            "B": "Queue",
            "C": "Heap",
            "D": "Hash function"
        },
        "correct_answer": "B",
        "explanation": "BFS uses a queue to explore vertices level by level."
    },
    {
        "question_id": "assess_dfs_01",
        "concept_id": "dfs",
        "question": "Which data structure is commonly used to implement iterative Depth-First Search?",
        "options": {
            "A": "Queue",
            "B": "Stack",
            "C": "Priority queue",
            "D": "Sorted array"
        },
        "correct_answer": "B",
        "explanation": "Iterative DFS uses a stack to explore deeper vertices first."
    }
]


def get_assessment_questions(concept_id=None):
    """
    Return assessment questions, optionally filtered by concept_id.
    Correct answers are not exposed to the student.
    """
    questions = QUESTION_BANK

    if concept_id is not None:
        questions = [
            q for q in questions
            if q["concept_id"] == concept_id
        ]

    return [
        {
            "question_id": q["question_id"],
            "concept_id": q["concept_id"],
            "question": q["question"],
            "options": q["options"]
        }
        for q in questions
    ]


def evaluate_assessment_answer(question_id, answer):
    """
    Evaluate an MCQ response.
    Accepts an option letter (A/B/C/D) or the exact option text.
    Returns a binary outcome suitable for BKT.
    """
    question = next(
        (q for q in QUESTION_BANK if q["question_id"] == question_id),
        None
    )

    if question is None:
        raise ValueError(f"Unknown question_id: {question_id}")

    if answer is None or not str(answer).strip():
        raise ValueError("Answer cannot be empty.")

    submitted = str(answer).strip().casefold()

    correct_option = question["correct_answer"]
    correct_text = question["options"][correct_option]

    valid_answers = {
        correct_option.casefold(),
        correct_text.casefold()
    }

    outcome = int(submitted in valid_answers)

    return {
        "question_id": question_id,
        "concept_id": question["concept_id"],
        "outcome": outcome,
        "classification": "correct" if outcome == 1 else "incorrect",
        "feedback": (
            "Correct! " + question["explanation"]
            if outcome == 1
            else "Incorrect. " + question["explanation"]
        )
    }
'''

assessment_path = DATA_DIR / "assessment.py"
assessment_path.write_text(assessment_code, encoding="utf-8")

print("Created:", assessment_path)

Created: /content/drive/MyDrive/LearnSyncAI/data/assessment.py


In [17]:
import sys

if str(DATA_DIR) not in sys.path:
    sys.path.append(str(DATA_DIR))

from assessment import (
    get_assessment_questions,
    evaluate_assessment_answer
)

# Check the full question bank
questions = get_assessment_questions()

print("Total assessment questions:", len(questions))
print("\nFirst question:")
print(questions[0])

# Test a correct response
correct_test = evaluate_assessment_answer(
    "assess_arrays_01", "A"
)

print("\nCorrect answer test:")
print(correct_test)

# Test an incorrect response
incorrect_test = evaluate_assessment_answer(
    "assess_arrays_01", "B"
)

print("\nIncorrect answer test:")
print(incorrect_test)

# Test concept-specific retrieval
stack_questions = get_assessment_questions("stacks")

print("\nStack questions:", len(stack_questions))

Total assessment questions: 13

First question:
{'question_id': 'assess_arrays_01', 'concept_id': 'arrays', 'question': 'What is the time complexity of accessing an element using its index in an array?', 'options': {'A': 'O(1)', 'B': 'O(n)', 'C': 'O(log n)', 'D': 'O(n^2)'}}

Correct answer test:
{'question_id': 'assess_arrays_01', 'concept_id': 'arrays', 'outcome': 1, 'classification': 'correct', 'feedback': 'Correct! Array elements can be accessed directly using their index.'}

Incorrect answer test:
{'question_id': 'assess_arrays_01', 'concept_id': 'arrays', 'outcome': 0, 'classification': 'incorrect', 'feedback': 'Incorrect. Array elements can be accessed directly using their index.'}

Stack questions: 1


In [18]:
from assessment import evaluate_assessment_answer
from bkt_engine import update_bkt, get_mastery_profile, reset_bkt_state

# Start with a clean test student
reset_bkt_state()

student_id = "test_student_001"

# Simulate answering an arrays question correctly
assessment_result = evaluate_assessment_answer(
    "assess_arrays_01", "A"
)

print("Assessment result:")
print(assessment_result)

# Pass the binary outcome to BKT
bkt_result = update_bkt(
    student_id=student_id,
    concept_id=assessment_result["concept_id"],
    outcome=assessment_result["outcome"]
)

print("\nBKT update result:")
print(bkt_result)

print("\nUpdated mastery profile:")
print(get_mastery_profile(student_id))

Assessment result:
{'question_id': 'assess_arrays_01', 'concept_id': 'arrays', 'outcome': 1, 'classification': 'correct', 'feedback': 'Correct! Array elements can be accessed directly using their index.'}

BKT update result:
{'student_id': 'test_student_001', 'concept_id': 'arrays', 'outcome': 1, 'prior_mastery': 0.2, 'predicted_correct_probability': 0.3400000000000001, 'posterior_mastery': 0.5294117647058822, 'mastery': 0.5764705882352941}

Updated mastery profile:
{'arrays': 0.5764705882352941}


In [19]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

report = r"""
# LearnSync AI — Knowledge Tracing Report
## Member 1: Dataset Processing, Assessment and Bayesian Knowledge Tracing

## 1. Overview

This component of LearnSync AI processes student programming-submission
interactions, maps programming problems to DSA concepts, builds chronological
student-concept sequences, estimates concept mastery using Bayesian Knowledge
Tracing (BKT), and provides an initial multiple-choice assessment interface.

The implementation uses Python, Pandas and a custom BKT engine.

## 2. Dataset Description

The project uses a Zenodo dataset containing Java programming submissions.

The raw submission dataset contains:
- 16,074 submission records
- 270 unique students
- 99 programming activities
- Submission outcomes, timestamps, activity identifiers and source code

The result field contains successful and failed submissions, as well as
records with empty results.

Preprocessing:
- Successful submissions were converted to outcome = 1.
- Failed submissions were converted to outcome = 0.
- Records with empty results were excluded from binary BKT observations.
- Missing timestamps were handled using available alternate timestamp fields.
- Interactions were ordered chronologically.

The resulting binary interaction dataset contains 15,474 observations.

## 3. DSA Concept Mapping

The 99 activities were mapped to the project's agreed DSA concept identifiers
using their problem statements.

The mapping includes:
- arrays
- linked_lists
- stacks
- queues
- recursion
- trees
- binary_search_trees
- heaps
- sorting
- hashing
- graphs
- bfs
- dfs
- unmapped

Two mappings were manually corrected during review:
- queue_jumper: queues -> arrays
- hash_table_chaining: queues -> hashing

Seven activities were retained as unmapped because their available problem
statements did not support a sufficiently confident DSA classification.

The final mapping contains 99 activities.

## 4. Preprocessing and Student Sequences

After joining the concept mapping with the preprocessed interactions,
14,390 observations were available for mapped concept-level knowledge tracing.

The sequence-building function groups observations by student and concept,
then orders attempts by timestamp.

Results:
- Students with mapped interactions: 270
- Student-concept sequences: 1,508
- Mapped observations: 14,390

Each observation contains:
- student_id
- concept_id
- question_id
- timestamp
- outcome (0 or 1)

Unmapped activities are excluded from concept-level BKT updates.

## 5. Bayesian Knowledge Tracing

Bayesian Knowledge Tracing estimates the probability that a student has
mastered a particular concept.

The implementation maintains a mastery probability for each
student-concept pair.

The baseline parameters are:

| Parameter | Value | Description |
|---|---:|---|
| P_INIT | 0.20 | Initial probability of mastery |
| P_LEARN | 0.10 | Probability of learning after an attempt |
| P_GUESS | 0.20 | Probability of a correct guess |
| P_SLIP | 0.10 | Probability of an incorrect answer despite mastery |

These are initial configurable assumptions. They were not fitted or
optimized using a held-out dataset.

The BKT engine supports:
- Updating mastery after correct and incorrect responses
- Retrieving a student's concept mastery profile
- Predicting the probability of a correct response
- Evaluating sequential one-step-ahead predictions
- Resetting in-memory mastery state

## 6. BKT Evaluation

The baseline BKT implementation was evaluated using the 14,390 mapped
chronological observations.

| Metric | Result |
|---|---:|
| Observations | 14,390 |
| Prediction accuracy | 0.6814 |
| Brier score | 0.2233 |
| Log loss | 0.6507 |

The evaluation predicts each response using the mastery estimate available
before that response is observed, then updates mastery using the observed
outcome.

These results represent baseline sequential prediction performance.
They should not be interpreted as held-out test performance or evidence
of generalization to unseen students.

## 7. Example Mastery Profile

After processing the chronological sequences, the following mastery
estimates were observed for user_0001:

| Concept | Estimated mastery |
|---|---:|
| Arrays | 0.1143 |
| Recursion | 0.8118 |
| Sorting | 0.9706 |
| Stacks | 0.9559 |
| Queues | 0.1169 |
| Hashing | 0.1161 |
| Binary Search Trees | 0.9997 |
| BFS | 0.5765 |
| Heaps | 0.7980 |

These are estimates for one example student, not population averages.

## 8. Assessment Component

An initial multiple-choice assessment question bank was implemented
in assessment.py.

It contains 13 questions, with one question for each of the 13 agreed
DSA concept identifiers.

The module provides:
- get_assessment_questions(concept_id=None)
- evaluate_assessment_answer(question_id, answer)

The answer evaluator returns a structured result containing:
- question_id
- concept_id
- outcome
- classification
- feedback

Correct responses produce outcome = 1, while incorrect responses produce
outcome = 0.

The assessment output was tested with the BKT engine. A correct response
was successfully passed to update_bkt(), resulting in an updated mastery
estimate.

## 9. Limitations

1. The assessment bank currently contains only one MCQ per concept.
   It is an initial functional question bank, not a comprehensive assessment.

2. BKT parameters are baseline assumptions and have not been optimized
   through parameter search or statistical fitting.

3. Evaluation uses sequential predictions over the available historical
   interaction sequences. A separate held-out evaluation is still needed
   to assess generalization.

4. Programming submission outcomes are binary proxies for correctness.
   They do not directly measure conceptual understanding.

5. Multiple submissions to the same question may be closely spaced in time
   and should not automatically be interpreted as independent learning events.

6. Seven activities remain unmapped because their available descriptions
   did not support confident classification.

7. The current BKT engine stores mastery in memory. Persistent storage
   is intended to be handled by the integrated database component.

## 10. Deliverables

Implemented:
- concept_mapping.csv
- preprocessed_interactions.csv
- bkt_engine.py
- assessment.py
- knowledge_tracing_report.md

The dataset analysis and preprocessing workflow is maintained in the
development notebook.

## 11. Conclusion

The Member 1 component establishes a working data-processing,
concept-mapping, assessment and knowledge-tracing pipeline for LearnSync AI.

The processed interaction data is converted into chronological
student-concept sequences, and the BKT engine updates mastery estimates
using binary assessment outcomes.

The assessment and BKT modules have also been tested together, providing
the interface needed for integration with the Learning Orchestrator.
"""

report_path = DATA_DIR / "knowledge_tracing_report.md"
report_path.write_text(report, encoding="utf-8")

print("Report created successfully:", report_path)
print("File size:", report_path.stat().st_size, "bytes")

Report created successfully: /content/drive/MyDrive/LearnSyncAI/data/knowledge_tracing_report.md
File size: 6879 bytes


In [20]:
#Temporary verification cells

In [21]:
import sys
import importlib
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

required_functions = {
    "assessment": [
        "get_assessment_questions",
        "evaluate_assessment_answer"
    ],
    "bkt_engine": [
        "update_bkt",
        "get_mastery_profile"
    ]
}

for module_name, functions in required_functions.items():
    module = importlib.import_module(module_name)
    print(f"\n{module_name}.py")

    for function_name in functions:
        print(
            function_name,
            "FOUND" if callable(getattr(module, function_name, None))
            else "MISSING"
        )

print("\nRequired files:")
for filename in [
    "concept_mapping.csv",
    "preprocessed_interactions.csv",
    "assessment.py",
    "bkt_engine.py",
    "knowledge_tracing_report.md"
]:
    print(filename, "FOUND" if (DATA_DIR / filename).exists() else "MISSING")

print("\nNotebook-only sequence function:")
print("build_student_sequences:",
      "FOUND" if "build_student_sequences" in globals() else "MISSING")


assessment.py
get_assessment_questions FOUND
evaluate_assessment_answer FOUND

bkt_engine.py
update_bkt FOUND
get_mastery_profile FOUND

Required files:
concept_mapping.csv FOUND
preprocessed_interactions.csv FOUND
assessment.py FOUND
bkt_engine.py FOUND
knowledge_tracing_report.md FOUND

Notebook-only sequence function:
build_student_sequences: FOUND


In [22]:
from pathlib import Path

bkt_path = DATA_DIR / "bkt_engine.py"

code = bkt_path.read_text(encoding="utf-8")

old_function = '''def get_mastery_profile(student_id):
    """Return the current mastery estimates for one student."""
    student_id = str(student_id)

    return {
        concept_id: mastery
        for (sid, concept_id), mastery in _mastery.items()
        if sid == student_id
    }
'''

new_function = '''def get_mastery_profile(student_id):
    """
    Return the student's mastery profile in the shared project format.

    Output:
        {
            "student_id": "...",
            "mastery": {"arrays": 0.72}
        }
    """
    student_id = str(student_id)

    mastery = {
        concept_id: value
        for (sid, concept_id), value in _mastery.items()
        if sid == student_id
    }

    return {
        "student_id": student_id,
        "mastery": mastery
    }
'''

if old_function not in code:
    raise ValueError(
        "Original function not found. File was not modified."
    )

bkt_path.write_text(
    code.replace(old_function, new_function),
    encoding="utf-8"
)

print("Updated get_mastery_profile() successfully.")

Updated get_mastery_profile() successfully.


In [23]:
import importlib
import bkt_engine

importlib.reload(bkt_engine)

bkt_engine.reset_bkt_state()

bkt_engine.update_bkt("student_001", "arrays", 1)
bkt_engine.update_bkt("student_001", "linked_lists", 0)

profile = bkt_engine.get_mastery_profile("student_001")

print(profile)

{'student_id': 'student_001', 'mastery': {'arrays': 0.5764705882352941, 'linked_lists': 0.1272727272727273}}


In [24]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

# Load the cleaned interactions and latest corrected mapping
interactions = pd.read_csv(DATA_DIR / "preprocessed_interactions_draft.csv")
mapping = pd.read_csv(DATA_DIR / "concept_mapping.csv")

# Check that the required columns exist
required_interaction_cols = {"question_id", "student_id", "timestamp", "outcome"}
required_mapping_cols = {
    "question_id", "concept_id", "mapping_reason", "confidence"
}

assert required_interaction_cols.issubset(interactions.columns), (
    f"Missing interaction columns: "
    f"{required_interaction_cols - set(interactions.columns)}"
)
assert required_mapping_cols.issubset(mapping.columns), (
    f"Missing mapping columns: "
    f"{required_mapping_cols - set(mapping.columns)}"
)

# Remove the old concept assignment before applying the corrected mapping
interactions = interactions.drop(columns=["concept_id"], errors="ignore")

# Ensure matching ID types
interactions["question_id"] = interactions["question_id"].astype(str)
mapping["question_id"] = mapping["question_id"].astype(str)

# Prevent duplicate question IDs from multiplying interaction rows
assert mapping["question_id"].is_unique, "Duplicate question IDs found in mapping!"

# Apply the latest concept mapping
final_interactions = interactions.merge(
    mapping[["question_id", "concept_id", "mapping_reason", "confidence"]],
    on="question_id",
    how="left",
    validate="many_to_one"
)

# Verify that every interaction received a mapping
assert final_interactions["concept_id"].notna().all(), (
    "Some interactions did not receive a concept mapping!"
)

# Save the corrected dataset
output_path = DATA_DIR / "preprocessed_interactions.csv"
final_interactions.to_csv(output_path, index=False)

print("UPDATED PREPROCESSED DATASET")
print("Total interactions:", len(final_interactions))
print("Unique students:", final_interactions["student_id"].nunique())
print("Unique questions:", final_interactions["question_id"].nunique())
print("\nInteractions per concept:")
print(final_interactions["concept_id"].value_counts())
print("\nSaved to:", output_path)

UPDATED PREPROCESSED DATASET
Total interactions: 15474
Unique students: 270
Unique questions: 99

Interactions per concept:
concept_id
sorting                4361
arrays                 2418
binary_search_trees    1666
queues                 1318
stacks                 1209
recursion              1114
unmapped               1084
hashing                1007
heaps                   455
linked_lists            385
bfs                     312
graphs                  124
trees                    21
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/LearnSyncAI/data/preprocessed_interactions.csv


In [25]:
import pandas as pd
from collections import defaultdict
from pathlib import Path
import importlib
import bkt_engine

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

# Load the corrected interactions
interactions = pd.read_csv(DATA_DIR / "preprocessed_interactions.csv")

def build_student_sequences(interactions):
    required_columns = {
        "student_id", "concept_id", "question_id", "timestamp", "outcome"
    }

    missing = required_columns - set(interactions.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    data = interactions.copy()
    data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
    data["outcome"] = pd.to_numeric(data["outcome"], errors="coerce")

    data = data[
        data["concept_id"].notna()
        & (data["concept_id"] != "unmapped")
        & data["outcome"].isin([0, 1])
        & data["timestamp"].notna()
    ].copy()

    data = data.sort_values(["student_id", "timestamp"])

    sequences = defaultdict(list)

    for row in data.itertuples(index=False):
        key = (str(row.student_id), row.concept_id)

        sequences[key].append({
            "question_id": row.question_id,
            "timestamp": row.timestamp.isoformat(),
            "outcome": int(row.outcome)
        })

    return dict(sequences)


# Build sequences using the corrected mapping
sequences = build_student_sequences(interactions)

print("SEQUENCE SUMMARY")
print("Student-concept sequences:", len(sequences))
print("Unique students:", len(set(student for student, concept in sequences)))
print("Mapped observations:", sum(len(seq) for seq in sequences.values()))

# Reload BKT module and evaluate from a clean state
importlib.reload(bkt_engine)

results = bkt_engine.evaluate_bkt(sequences)

print("\nUPDATED BKT EVALUATION")
print(results)

# Display one student's mastery profile
student_id = "user_0001"
print("\nEXAMPLE MASTERY PROFILE")
print(bkt_engine.get_mastery_profile(student_id))

SEQUENCE SUMMARY
Student-concept sequences: 1491
Unique students: 270
Mapped observations: 14390

UPDATED BKT EVALUATION
{'observations': 14390, 'accuracy': 0.6833217512161223, 'brier_score': 0.2227860257102827, 'log_loss': 0.6497797593222633}

EXAMPLE MASTERY PROFILE
{'student_id': 'user_0001', 'mastery': {'arrays': 0.11432102524959908, 'recursion': 0.8117647058823529, 'sorting': 0.9706093376811333, 'stacks': 0.9558958652373661, 'queues': 0.5764705882352941, 'hashing': 0.11429061875462478, 'binary_search_trees': 0.9996684052364342, 'bfs': 0.5764705882352941, 'heaps': 0.7980128579777908}}


In [26]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")
report_path = DATA_DIR / "knowledge_tracing_report.md"

report = """
# LearnSync AI — Knowledge Tracing Report
## Member 1: Assessment and Knowledge Tracing

## 1. Objective

The objective of this module is to analyse student programming submissions,
map programming problems to DSA concepts, construct chronological student
learning sequences, assess student knowledge using objective questions, and
estimate concept mastery using Bayesian Knowledge Tracing (BKT).

## 2. Dataset Overview

The project uses the Zenodo programming-submission dataset.

Files inspected:
- java_submissions.csv
- distribution_per_question.csv
- tst_questions.csv

Key dataset observations:
- Raw Java submission records: 16,074
- Cleaned interactions: 15,474
- Unique students: 270
- Unique programming questions: 99
- Empty-result records excluded during preprocessing: 600

Submission outcomes:
- Successful submissions are represented as 1.
- Failed submissions are represented as 0.
- Empty-result records are excluded from binary BKT observations.

Missing timestamps were handled using the available alternative timestamp
fields during preprocessing.

## 3. Concept Mapping

Programming questions were mapped to the project's agreed DSA concept IDs.

The mapping includes:
- concept_id
- mapping_reason
- confidence

Seven questions remain unmapped because their primary DSA concept could not
be assigned confidently.

Important mapping corrections:
- hash_table_chaining -> hashing
- queue_jumper -> arrays

Unmapped questions are retained in the preprocessed dataset but excluded
from concept-specific BKT sequences.

## 4. Preprocessing and Student Sequences

The final preprocessed dataset contains:
- 15,474 interactions
- 270 unique students
- 99 unique questions

Interactions are grouped by student and concept, then ordered
chronologically by timestamp.

Final sequence statistics:
- Student-concept sequences: 1,491
- Unique students with mapped observations: 270
- Mapped observations used by BKT: 14,390

## 5. Assessment Module

The assessment module provides 13 objective multiple-choice questions,
covering the agreed DSA concepts.

Implemented functions:
- get_assessment_questions(concept_id=None)
- evaluate_assessment_answer(question_id, answer)

The evaluator returns:
- question_id
- concept_id
- binary outcome (0 or 1)
- correctness classification
- feedback and explanation

The correct answer is not exposed by the question-retrieval function.

The current question bank is a prototype with one question per concept,
not a comprehensive assessment bank.

## 6. Bayesian Knowledge Tracing

BKT estimates the probability that a student has mastered a concept.

Initial baseline parameters:
- Initial mastery (P_INIT): 0.20
- Learning probability (P_LEARN): 0.10
- Guess probability (P_GUESS): 0.20
- Slip probability (P_SLIP): 0.10

These are baseline assumptions and have not been fitted or optimized
against a held-out dataset.

Implemented functions:
- update_bkt(student_id, concept_id, outcome)
- get_mastery_profile(student_id)
- predict_correct_probability(...)
- evaluate_bkt(sequences)
- reset_bkt_state()

The mastery profile follows the shared project contract:

{
    "student_id": "student_001",
    "mastery": {
        "arrays": 0.72,
        "linked_lists": 0.45
    }
}

Mastery estimates are bounded between 0 and 1.

## 7. BKT Evaluation

Evaluation was performed by processing each student's mapped interactions
in chronological order. For each observation, the model predicts the
probability of a correct response before updating its mastery estimate
using the observed outcome.

Final evaluation results:

| Metric | Result |
|---|---:|
| Observations | 14,390 |
| Accuracy | 0.6833 |
| Brier score | 0.2228 |
| Log loss | 0.6498 |

These are sequential one-step-ahead results on the available interaction
sequences. They should not be interpreted as held-out test performance
or evidence of generalization to unseen students.

## 8. Example Mastery Profile

Example student: user_0001

| Concept | Estimated mastery |
|---|---:|
| Arrays | 0.1143 |
| Recursion | 0.8118 |
| Sorting | 0.9706 |
| Stacks | 0.9559 |
| Queues | 0.5765 |
| Hashing | 0.1143 |
| Binary Search Trees | 0.9997 |
| BFS | 0.5765 |
| Heaps | 0.7980 |

The profile illustrates how mastery estimates are maintained separately
for each student and DSA concept.

## 9. Limitations

1. BKT parameters are baseline assumptions, not learned parameters.
2. Evaluation uses sequential historical observations and does not use
   a held-out test set.
3. Programming submission outcomes are proxies for correctness and do
   not directly prove conceptual understanding.
4. Repeated submissions may be correlated.
5. Seven questions remain unmapped.
6. The assessment bank currently contains only one question per concept.
7. The current BKT implementation maintains state in memory; persistent
   storage is handled by the integration layer.

## 10. Conclusion

The Member 1 module provides a working data preprocessing pipeline,
documented problem-to-concept mappings, chronological student-concept
sequences, an objective assessment bank, and a functional BKT engine.

The module produces per-student, per-concept mastery estimates in the
shared project format and has been evaluated on 14,390 mapped observations.
"""

report_path.write_text(report.strip() + "\n", encoding="utf-8")

print("Report updated successfully.")
print("Path:", report_path)
print("Size:", report_path.stat().st_size, "bytes")

Report updated successfully.
Path: /content/drive/MyDrive/LearnSyncAI/data/knowledge_tracing_report.md
Size: 5244 bytes


In [28]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")
module_path = DATA_DIR / "data_pipeline.py"

module_code = r'''
from pathlib import Path
from collections import defaultdict

import pandas as pd


def load_zenodo_data(data_directory):
    """
    Load the three Zenodo dataset tables.

    Returns a dictionary containing:
    submissions, distribution, and questions.
    """
    data_directory = Path(data_directory)

    submissions_path = data_directory / "java_submissions.csv"
    distribution_path = data_directory / "distribution_per_question.csv"
    questions_path = data_directory / "tst_questions.csv"

    for path in [submissions_path, distribution_path, questions_path]:
        if not path.exists():
            raise FileNotFoundError(f"Dataset file not found: {path}")

    return {
        "submissions": pd.read_csv(submissions_path),
        "distribution": pd.read_csv(distribution_path),
        "questions": pd.read_csv(questions_path, sep="~")
    }


def _parse_timestamp_column(series):
    """Parse timestamps stored as epoch milliseconds or date strings."""
    numeric = pd.to_numeric(series, errors="coerce")
    parsed_numeric = pd.to_datetime(
        numeric, unit="ms", errors="coerce"
    )

    parsed_text = pd.to_datetime(series, errors="coerce")

    return parsed_numeric.fillna(parsed_text)


def preprocess_interactions(raw_data):
    """
    Convert raw Java submissions into cleaned student interactions.

    Success -> outcome 1
    Fail    -> outcome 0
    Empty or unrecognized results are excluded.

    Missing primary timestamps are filled from available alternatives.
    """
    if "submissions" not in raw_data:
        raise KeyError("raw_data must contain a 'submissions' table")

    df = raw_data["submissions"].copy()

    required = {"user", "activity", "result", "timestamp"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Missing submission columns: {missing}")

    df["student_id"] = df["user"].astype(str)
    df["question_id"] = df["activity"].astype(str)

    result = df["result"].fillna("").astype(str).str.strip().str.lower()

    df["outcome"] = pd.NA
    df.loc[result.str.startswith("success"), "outcome"] = 1
    df.loc[result.str.startswith("fail"), "outcome"] = 0

    # Parse primary timestamps, then use fallback columns where needed.
    df["timestamp"] = _parse_timestamp_column(df["timestamp"])
    df["timestamp_source"] = "timestamp"

    for column in ["timisOrderedmp", "timismp"]:
        if column in df.columns:
            fallback = _parse_timestamp_column(df[column])
            missing_time = df["timestamp"].isna() & fallback.notna()

            df.loc[missing_time, "timestamp"] = fallback[missing_time]
            df.loc[missing_time, "timestamp_source"] = column

    # Keep only valid binary outcomes and usable timestamps.
    df = df[
        df["outcome"].notna() &
        df["timestamp"].notna()
    ].copy()

    df["outcome"] = df["outcome"].astype(int)

    output_columns = [
        "student_id",
        "question_id",
        "timestamp",
        "outcome"
    ]

    for column in ["result", "version", "timestamp_source"]:
        if column in df.columns:
            output_columns.append(column)

    return (
        df[output_columns]
        .sort_values(["student_id", "timestamp"])
        .reset_index(drop=True)
    )


def map_problems_to_concepts(problem_data, mapping_data=None):
    """
    Validate and apply the reviewed problem-to-concept mapping.

    If mapping_data is supplied, problem_data is merged with it by
    question_id. Otherwise, problem_data must already contain concept_id.

    This function does not invent or infer concept assignments.
    """
    required = {
        "question_id", "concept_id", "mapping_reason", "confidence"
    }

    if mapping_data is not None:
        missing = {"question_id"} - set(problem_data.columns)
        if missing:
            raise ValueError(f"Problem data missing columns: {missing}")

        missing_mapping = required - set(mapping_data.columns)
        if missing_mapping:
            raise ValueError(
                f"Mapping data missing columns: {missing_mapping}"
            )

        if mapping_data["question_id"].duplicated().any():
            raise ValueError("Mapping data contains duplicate question IDs")

        result = problem_data.drop(
            columns=["concept_id", "mapping_reason", "confidence"],
            errors="ignore"
        ).merge(
            mapping_data[list(required)],
            on="question_id",
            how="left",
            validate="many_to_one"
        )

        if result["concept_id"].isna().any():
            raise ValueError("Some questions have no concept mapping")

        return result

    missing = required - set(problem_data.columns)
    if missing:
        raise ValueError(
            f"Problem data must contain reviewed mapping columns: {missing}"
        )

    if problem_data["question_id"].duplicated().any():
        raise ValueError("Mapping data contains duplicate question IDs")

    return problem_data.copy()


def build_student_sequences(interactions):
    """
    Build chronological sequences grouped by student and DSA concept.

    Unmapped interactions and invalid outcomes are excluded.
    Returns a dictionary keyed by (student_id, concept_id).
    """
    required = {
        "student_id", "concept_id", "question_id", "timestamp", "outcome"
    }

    missing = required - set(interactions.columns)
    if missing:
        raise ValueError(f"Missing interaction columns: {missing}")

    data = interactions.copy()
    data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
    data["outcome"] = pd.to_numeric(data["outcome"], errors="coerce")

    data = data[
        data["concept_id"].notna() &
        (data["concept_id"] != "unmapped") &
        data["outcome"].isin([0, 1]) &
        data["timestamp"].notna()
    ].copy()

    data["student_id"] = data["student_id"].astype(str)
    data = data.sort_values(["student_id", "timestamp"])

    sequences = defaultdict(list)

    for row in data.itertuples(index=False):
        key = (row.student_id, row.concept_id)

        sequences[key].append({
            "question_id": row.question_id,
            "timestamp": row.timestamp.isoformat(),
            "outcome": int(row.outcome)
        })

    return dict(sequences)
'''

module_path.write_text(module_code.strip() + "\n", encoding="utf-8")

print("Created:", module_path)
print("Size:", module_path.stat().st_size, "bytes")


Created: /content/drive/MyDrive/LearnSyncAI/data/data_pipeline.py
Size: 6312 bytes


In [29]:
import sys
import importlib

if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

import data_pipeline
importlib.reload(data_pipeline)

# Load the original Zenodo files
raw_data = data_pipeline.load_zenodo_data(DATA_DIR)

# Preprocess the raw submissions
cleaned = data_pipeline.preprocess_interactions(raw_data)

# Load the reviewed mapping and apply it
mapping = pd.read_csv(DATA_DIR / "concept_mapping.csv")
mapped = data_pipeline.map_problems_to_concepts(cleaned, mapping)

# Build chronological sequences
sequences = data_pipeline.build_student_sequences(mapped)

print("MODULE TEST RESULTS")
print("Raw submission rows:", len(raw_data["submissions"]))
print("Cleaned interactions:", len(cleaned))
print("Mapped interactions:", len(mapped))
print("Student-concept sequences:", len(sequences))
print("Unique students:", mapped["student_id"].nunique())
print("\nAll four pipeline functions imported and executed.")

MODULE TEST RESULTS
Raw submission rows: 16074
Cleaned interactions: 15474
Mapped interactions: 15474
Student-concept sequences: 1491
Unique students: 270

All four pipeline functions imported and executed.


In [31]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/LearnSyncAI/data")

print("FILES IN DATA FOLDER:\n")

for path in sorted(DATA_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name}  |  {path.stat().st_size:,} bytes")

FILES IN DATA FOLDER:

assessment.py  |  7,878 bytes
bkt_engine.py  |  4,295 bytes
concept_mapping.csv  |  8,025 bytes
concept_mapping_draft.csv  |  124,680 bytes
concept_mapping_review.csv  |  198,823 bytes
concept_mapping_review_v2.csv  |  199,004 bytes
concept_mapping_review_v3.csv  |  198,982 bytes
data_pipeline.py  |  6,312 bytes
distribution_per_question.csv  |  2,005 bytes
java_submissions.csv  |  27,859,931 bytes
knowledge_tracing_report.md  |  5,244 bytes
preprocessed_interactions.csv  |  2,170,135 bytes
preprocessed_interactions_draft.csv  |  1,358,026 bytes
tst_questions.csv  |  192,830 bytes


In [33]:
import importlib
import assessment
import bkt_engine

importlib.reload(assessment)
importlib.reload(bkt_engine)

# Start with clean BKT state
bkt_engine.reset_bkt_state()

# Retrieve the arrays question
question = assessment.get_assessment_questions("arrays")[0]

question_id = question["question_id"]

# Access the answer internally for testing only
answer_key = next(
    q for q in assessment.QUESTION_BANK
    if q["question_id"] == question_id
)

correct_answer = answer_key["correct_answer"]

# Evaluate the answer
evaluation = assessment.evaluate_assessment_answer(
    question_id,
    correct_answer
)

# Send assessment outcome to BKT
update = bkt_engine.update_bkt(
    "integration_test_student",
    evaluation["concept_id"],
    evaluation["outcome"]
)

profile = bkt_engine.get_mastery_profile(
    "integration_test_student"
)

print("ASSESSMENT → BKT INTEGRATION")
print("Question:", question_id)
print("Assessment result:", evaluation["classification"])
print("Outcome:", evaluation["outcome"])
print("BKT mastery update:", update["mastery"])
print("Mastery profile:", profile)

assert evaluation["outcome"] == 1
assert profile["student_id"] == "integration_test_student"
assert "arrays" in profile["mastery"]
assert 0 <= profile["mastery"]["arrays"] <= 1

print("\nINTEGRATION TEST PASSED")

ASSESSMENT → BKT INTEGRATION
Question: assess_arrays_01
Assessment result: correct
Outcome: 1
BKT mastery update: 0.5764705882352941
Mastery profile: {'student_id': 'integration_test_student', 'mastery': {'arrays': 0.5764705882352941}}

INTEGRATION TEST PASSED


In [34]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/LearnSyncAI")

print("Searching for notebooks inside LearnSyncAI...\n")

notebooks = list(ROOT.rglob("*.ipynb"))

if notebooks:
    for notebook in notebooks:
        print(notebook)
else:
    print("No notebooks found inside LearnSyncAI.")

Searching for notebooks inside LearnSyncAI...

/content/drive/MyDrive/LearnSyncAI/Member_1/dataset_analysis.ipynb
/content/drive/MyDrive/LearnSyncAI/Member_2/rag_development.ipynb


In [35]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/LearnSyncAI")

deliverables = {
    "Dataset analysis notebook": ROOT / "Member_1/dataset_analysis.ipynb",
    "Data pipeline": ROOT / "data/data_pipeline.py",
    "Assessment module": ROOT / "data/assessment.py",
    "BKT engine": ROOT / "data/bkt_engine.py",
    "Concept mapping": ROOT / "data/concept_mapping.csv",
    "Preprocessed interactions": ROOT / "data/preprocessed_interactions.csv",
    "Knowledge tracing report": ROOT / "data/knowledge_tracing_report.md"
}

print("FINAL MEMBER 1 DELIVERABLES\n")

all_found = True

for name, path in deliverables.items():
    exists = path.exists()
    print(f"{'FOUND' if exists else 'MISSING'} — {name}")
    all_found = all_found and exists

print("\nFINAL STATUS:", "ALL FILES PRESENT" if all_found else "FILES MISSING")

FINAL MEMBER 1 DELIVERABLES

FOUND — Dataset analysis notebook
FOUND — Data pipeline
FOUND — Assessment module
FOUND — BKT engine
FOUND — Concept mapping
FOUND — Preprocessed interactions
FOUND — Knowledge tracing report

FINAL STATUS: ALL FILES PRESENT
